# 01 — Feature extraction

Build detector-ready features for all supported feature views. The feature table contains runtime (`rt_`), process (`proc_`), and controller (`ctrl_`) columns. Model training later selects feature subsets by prefix.

In [1]:
from pathlib import Path
from robustedge.pipeline import load_config, build_features
from robustedge.features import infer_feature_columns

DATA_ROOT = Path('../data/raw/logs')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('../tests/fixtures/minimal_dataset')

OUTPUT_DIR = Path('../outputs/notebook_run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = load_config('../configs/default.yaml')
features, manifest, runs = build_features(DATA_ROOT, config)
features.to_csv(OUTPUT_DIR / 'features.csv', index=False)
manifest.to_csv(OUTPUT_DIR / 'manifest.csv', index=False)

print(features.shape)
print('runtime cols', len(infer_feature_columns(features, prefixes=('rt_',))))
print('process cols', len(infer_feature_columns(features, prefixes=('proc_',))))
print('controller cols', len(infer_feature_columns(features, prefixes=('ctrl_',))))
features.head()

(45101, 315)
runtime cols 70
process cols 202
controller cols 25


,relative_time_s,window_start,window_end,timestamp,rt_lstat,rt_renameat,rt_unlinkat,rt_procinfo,rt_sched_yield,rt_close,...,label,rt_madvise,rt_rseq,rt_clone3,rt_setgid,rt_setuid,rt_mkdirat,rt_getdents64,rt_lseek,rt_tgkill
0,0.000000,0.000000,4.000000,2026-04-24 19:20:48.407564032,4.0,4.0,8.0,112,33.0,8.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3.783155,3.783155,7.783155,2026-04-24 19:20:52.190718976,0.0,0.0,0.0,93,0.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7.930423,7.930423,11.930423,2026-04-24 19:20:56.337987072,0.0,0.0,0.0,96,6.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,12.331006,12.331006,16.331006,2026-04-24 19:21:00.738569984,0.0,0.0,0.0,96,4.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,16.202436,16.202436,20.202436,2026-04-24 19:21:04.609999872,0.0,0.0,0.0,98,28.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
# Check attack labels and campaign phases
features.groupby(['phase', 'attack_duration', 'perturbation_family', 'severity'], dropna=False)['label'].agg(['sum', 'count']).reset_index().head(30)

,phase,attack_duration,perturbation_family,severity,sum,count
0,phase1_clean_benign,0.0,none,0.0,0,1323
1,phase2_clean_attacked,20.0,none,0.0,43,1327
2,phase2_clean_attacked,100.0,none,0.0,217,1328
3,phase2_clean_attacked,300.0,none,0.0,657,1328
4,phase3_perturbed_benign,0.0,P1,0.5,0,1326
5,phase3_perturbed_benign,0.0,P1,1.0,0,1326
6,phase3_perturbed_benign,0.0,P2,0.5,0,1328
7,phase3_perturbed_benign,0.0,P2,1.0,0,1325
8,phase3_perturbed_benign,0.0,P3,0.5,0,1326
9,phase3_perturbed_benign,0.0,P3,1.0,0,1326
